# ML common XGBoost Native Categorical Run

`ml_inputs`에 승인 배치된 experiment 입력을 사용해 XGBoost 학습/validation 흐름을 실행한다.


## 실행 범위

- `fb_outputs` 검토 후 `ml_inputs/{RUN_ID}`에 승인 배치된 `{ARTIFACT_PREFIX}` 입력을 사용한다.
- `encoding_manifest.json`을 `XGBTrainConfig`에 전달해 native categorical dtype을 복원한다.
- train split으로 모델을 학습하고 validation split에서 threshold와 metric을 산출한다.
- feature 생성이나 `fb_outputs -> ml_inputs` 복사는 수행하지 않는다.
- exp 식별자는 설정 셀의 `EXPORT_EXPERIMENT_ID`, `RUN_ID`, `MODEL_RUN_ID`에서만 지정한다.
- final test는 기본 실행하지 않는다. full validation artifact 확정 후 사용자가 `RUN_FINAL_TEST=True`로 전환해 실행한다.


## 실행 전 주의

- 입력 feature set은 `*_feature_contract_approve.csv`에서 `used_in_ml=TRUE`인 컬럼을 기준으로 한다.
- native categorical dtype 복원에는 같은 run directory의 `encoding_manifest.json`이 필요하다.
- threshold 선택은 validation set에서만 수행하고, test set은 `RUN_FINAL_TEST=True`일 때만 사용한다.
- 기본 overwrite는 `False`다. 같은 설정으로 재실행하려면 `MODEL_RUN_ID` 변경을 권장한다.

## Optuna best 모델 선택 워크플로
- `RUN_TUNING=True`는 smoke/full tuning을 실행한 뒤 full study의 best params로 root `ml_outputs/<RUN_ID>/`에 ML-01과 동일한 최종 산출물 세트를 만든다.
- root 최종 산출물만 validation 비교와 리더보드 후보로 사용한다. `optuna__<MODEL_RUN_ID>/` 아래 smoke/full/best 로그는 참고용이며 필요하면 수기로 옮긴다.
- validation 결과가 마음에 들지 않으면 `MODEL_RUN_ID`를 바꿔 다시 Optuna를 실행한다. 예: `d01-optuna-ap`, `d02-optuna-f1`, `d03-optuna-ap-100trials`.
- validation 결과가 마음에 드는 `MODEL_RUN_ID` 하나를 최종 후보로 확정한 뒤, 그 `MODEL_RUN_ID`로 final test를 1회만 실행한다.
- final test는 best params를 직접 읽지 않는다. Optuna가 이미 best params로 만든 root `_model.pkl`, `_train_summary.json`, `_threshold.json`을 읽으므로 best param이 자동 반영된다.
- final test 단계에서는 학습/검증 셀을 다시 실행하지 않는다. 설정 셀에서 `MODEL_RUN_ID`, `RUN_FINAL_TEST=True`를 맞춘 뒤 `03_final test 잠금` 셀만 실행한다.
- test 결과를 보고 다시 `MODEL_RUN_ID`를 고르면 test leakage가 된다. test는 최종 후보 1개에 대해서만 사용한다.

# 00_경로 및 환경설정

In [1]:
from __future__ import annotations
import sys
from pathlib import Path

# ============================================================
# 0.1 Runtime Settings
# 실행 환경: ML 담당자 기준
# - Kernel          : Local WSL
# - Code repo       : local Git repo
# - Input artifacts : local Git repo ml/<exp-id>/ml_inputs/
# - Output artifacts: local Git repo ml/<exp-id>/ml_outputs/
# ============================================================
SEED = 42
EXPORT_EXPERIMENT_ID = "ml_03"
RUN_ID = "r00"
MODEL_RUN_ID = "d00-fixparam"

# Root Paths: 다른 환경에서 실행할 경우 보통 이 경로만 수정한다.
LOCAL_REPO_ROOT = Path("/mnt/d/AML_git/Graph_AML").resolve()

# Input/Output Paths
BASE_DIR = LOCAL_REPO_ROOT / "ml" / EXPORT_EXPERIMENT_ID.replace("_", "-")
ML_INPUTS_DIR = BASE_DIR / "ml_inputs"
ML_OUTPUTS_DIR = BASE_DIR / "ml_outputs"

# Module Code Paths: local Git repo에서 train_val_test 모듈을 import한다.
MODULE_DIR = BASE_DIR / "01_train_val_test"

# ============================================================
# 0.2 Path Validation
# ============================================================
def require_dir(path: Path, name: str) -> None:
    # 필수 디렉터리가 없으면 시작 단계에서 중단한다.
    path = Path(path).resolve()
    if not path.exists():
        raise FileNotFoundError(f"{name} not found: {path}")
    if not path.is_dir():
        raise NotADirectoryError(f"{name} is not a directory: {path}")

# 시작 단계에서 반드시 존재해야 하는 디렉터리만 검증한다.
for name, path in {
    "LOCAL_REPO_ROOT": LOCAL_REPO_ROOT,
    "BASE_DIR": BASE_DIR,
    "ML_INPUTS_DIR": ML_INPUTS_DIR,
    "ML_OUTPUTS_DIR": ML_OUTPUTS_DIR,
    "MODULE_DIR": MODULE_DIR,
}.items():
    require_dir(path, name)

# ============================================================
# 0.3 Import Path Setup
# ============================================================
# local train_val_test/ml_*.py 모듈을 우선 import한다.
IMPORT_DIRS = [str(MODULE_DIR)]
IMPORT_MODULES = (
    "ml_cpugpu",
    "ml_utils",
    "ml_io",
    "ml_metrics",
    "ml_resource",
    "ml_optuna_spaces",
    "ml_train",
    "ml_val",
    "ml_test",
    "ml_optuna",
)

# 중복 경로를 제거한 뒤 맨 앞에 삽입해 import 우선순서를 고정한다.
sys.path = [path for path in sys.path if path not in IMPORT_DIRS]
sys.path[:0] = IMPORT_DIRS

# 노트북 재실행 시 수정된 local 모듈을 다시 읽도록 import cache를 제거한다.
for module_name in IMPORT_MODULES:
    sys.modules.pop(module_name, None)

import ml_cpugpu
import ml_utils
import ml_io
import ml_train
import ml_val
import ml_test
import ml_optuna

# train/validation/test 실행 전 난수 시드를 고정한다.
ml_utils.set_seed(SEED)

# ============================================================
# 0.4 Configuration Summary
# ============================================================
print("=" * 80)
print("SEED                 :", SEED)
print("EXPORT_EXPERIMENT_ID :", EXPORT_EXPERIMENT_ID)
print("RUN_ID               :", RUN_ID)
print("MODEL_RUN_ID         :", MODEL_RUN_ID)
print("LOCAL_REPO_ROOT      :", LOCAL_REPO_ROOT)
print("ML_INPUTS_DIR        :", ML_INPUTS_DIR)
print("ML_OUTPUTS_DIR       :", ML_OUTPUTS_DIR)
print("MODULE_DIR           :", MODULE_DIR)
print("=" * 80)


SEED                 : 42
EXPORT_EXPERIMENT_ID : ml_03
RUN_ID               : r00
MODEL_RUN_ID         : d00-fixparam
LOCAL_REPO_ROOT      : /mnt/d/AML_git/Graph_AML
ML_INPUTS_DIR        : /mnt/d/AML_git/Graph_AML/ml/ml-03/ml_inputs
ML_OUTPUTS_DIR       : /mnt/d/AML_git/Graph_AML/ml/ml-03/ml_outputs
MODULE_DIR           : /mnt/d/AML_git/Graph_AML/ml/ml-03/01_train_val_test


# 01_실행 설정

In [2]:
# ============================================================
# 실행 예시
# ============================================================
# 예시 1. 고정 XGB_PARAMS full train/validation 실행
# - encoding manifest가 있는 승인 입력으로 모델 학습 + validation threshold 선택
# - final test는 아직 실행하지 않음
#   RUN_TUNING = False
#   SAMPLE_ROWS = None
#   RUN_FINAL_TEST = False
#
# 예시 2. Optuna tuning + best params final train/validation 실행
# - Optuna는 test split을 절대 호출하지 않음
#   RUN_TUNING = True
#   RUN_FINAL_TEST = False
#
# 예시 3. 기존 full artifact로 final test만 실행
# - 같은 OUT_DIR에 prefix 기반 model, feature_columns, train_summary, threshold 파일이 있을 때만 실행
# - train/validation은 다시 돌리지 않고 final test만 수행
#   RUN_FINAL_TEST = True
#
# 주의:
# - EXPORT_EXPERIMENT_ID / RUN_ID / MODEL_RUN_ID는 00_경로 및 환경설정 셀에서만 지정한다.
# - RUN_ID는 fb_outputs 검토 후 ml_inputs에 승인 배치된 입력 묶음 식별자다.
# - MODEL_RUN_ID는 같은 입력으로 반복하는 모델 학습/검증 시도 식별자다.
# - sample/full 구분은 RUN_ID와 JSON metadata의 sampled, sample_rows 필드로 관리한다.
# - OVERWRITE_OUTPUTS=False이면 기존 산출물이 있을 때 덮어쓰지 않고 중단한다.

# ============================================================
# 산출물 구분
# ============================================================
# <ML_ARTIFACT_PREFIX>_model.pkl                    -> train 결과
# <ML_ARTIFACT_PREFIX>_feature_columns.json          -> train 결과
# <ML_ARTIFACT_PREFIX>_train_summary.json            -> train 결과 + native categorical 설정 기록
# <ML_ARTIFACT_PREFIX>_scores_train_summary.json     -> train/val logloss learning curve
# <ML_ARTIFACT_PREFIX>_feature_importance.csv         -> train 결과, booster importance
# <ML_ARTIFACT_PREFIX>_threshold.json                -> validation 결과
# <ML_ARTIFACT_PREFIX>_metrics_val.json              -> validation 결과
# <ML_ARTIFACT_PREFIX>_confusion_matrix_val.csv      -> validation 결과
# <ML_ARTIFACT_PREFIX>_prediction_scores_val.parquet -> validation row별 score, EXPORT_PREDICTION_SCORES=True일 때
# feature association JSON                          -> EXPORT_FEATURE_ASSOC=True일 때 train/val/test 단계별 생성
# <ML_ARTIFACT_PREFIX>_metrics_test.json             -> final test 결과
# <ML_ARTIFACT_PREFIX>_confusion_matrix_test.csv     -> final test 결과

# ============================================================
# 1.1 Run identifiers
# ============================================================
ARTIFACT_PREFIX = f"{EXPORT_EXPERIMENT_ID}__{RUN_ID}"
ML_ARTIFACT_PREFIX = f"{ARTIFACT_PREFIX}__{MODEL_RUN_ID}"

# ============================================================
# 1.2 Input / output directories
# ============================================================
# ML_INPUT_DIR은 사람이 검토 후 승인 배치한 ML 입력 위치다.
# OUT_DIR은 train/validation/test artifact를 저장하는 최종 run directory다.
ML_INPUT_DIR = ML_INPUTS_DIR / RUN_ID
OUT_DIR = ML_OUTPUTS_DIR / RUN_ID
OPTUNA_OUTPUT_DIR = OUT_DIR / f"optuna__{MODEL_RUN_ID}"

# ============================================================
# 1.3 Input artifact paths
# ============================================================
# train/val/test parquet와 승인 feature contract는 같은 ARTIFACT_PREFIX를 사용한다.
# ENCODING_MANIFEST_PATH는 native categorical dtype 복원과 검증에 사용된다.
TRAIN_PATH = ML_INPUT_DIR / f"{ARTIFACT_PREFIX}_Xy_train.parquet"
VAL_PATH = ML_INPUT_DIR / f"{ARTIFACT_PREFIX}_Xy_val.parquet"
TEST_PATH = ML_INPUT_DIR / f"{ARTIFACT_PREFIX}_Xy_test.parquet"
FEATURE_COLUMNS_PATH = ML_INPUT_DIR / f"{ARTIFACT_PREFIX}_fb_output_feature_contract_approve.csv"
ENCODING_MANIFEST_PATH = ML_INPUT_DIR / f"{ARTIFACT_PREFIX}_encoding_manifest.json"

MODEL_FILE_NAME = f"{ML_ARTIFACT_PREFIX}_model.pkl"
FEATURE_COLUMNS_FILE_NAME = f"{ML_ARTIFACT_PREFIX}_feature_columns.json"
TRAIN_SUMMARY_FILE_NAME = f"{ML_ARTIFACT_PREFIX}_train_summary.json"
SCORES_TRAIN_SUMMARY_FILE_NAME = f"{ML_ARTIFACT_PREFIX}_scores_train_summary.json"
FEATURE_IMPORTANCE_FILE_NAME = f"{ML_ARTIFACT_PREFIX}_feature_importance.csv"
THRESHOLD_FILE_NAME = f"{ML_ARTIFACT_PREFIX}_threshold.json"
METRICS_VAL_FILE_NAME = f"{ML_ARTIFACT_PREFIX}_metrics_val.json"
CONFUSION_MATRIX_VAL_FILE_NAME = f"{ML_ARTIFACT_PREFIX}_confusion_matrix_val.csv"
METRICS_TEST_FILE_NAME = f"{ML_ARTIFACT_PREFIX}_metrics_test.json"
CONFUSION_MATRIX_TEST_FILE_NAME = f"{ML_ARTIFACT_PREFIX}_confusion_matrix_test.csv"

# ============================================================
# 1.4 Execution switches
# ============================================================
# RUN_TUNING=False이면 고정 XGB_PARAMS, True이면 Optuna best params로 train/validation을 실행한다.
# SAMPLE_ROWS는 고정 XGB_PARAMS smoke/debug 전용이다. Optuna full tuning에서는 항상 None이어야 한다.
# final test는 full model + validation threshold 확정 후 명시적으로 True로 바꾼다.
# EXPORT_FEATURE_ASSOC는 train/val/test feature association JSON 생성 여부를 제어한다.
# EXPORT_PREDICTION_SCORES는 validation row별 score parquet 생성 여부를 제어한다.
SAMPLE_ROWS = None
RUN_TUNING = False
RUN_FINAL_TEST = False
OVERWRITE_OUTPUTS = False

EXPORT_FEATURE_ASSOC = True
EXPORT_PREDICTION_SCORES = True

XGB_ACCELERATOR = "auto"  # auto | cpu | cuda

OPTUNA_SMOKE_TRIALS = 3
OPTUNA_FULL_TRIALS = 25
OPTUNA_SELECTION_METRIC = "f1"

# ============================================================
# 1.5 XGBoost parameters
# ============================================================

XGB_PARAMS = {
    "n_estimators": 300,
    "max_depth": 4,
    "learning_rate": 0.05,
    "subsample": 0.9,
    "colsample_bytree": 0.9,
    "min_child_weight": 1.0,
    "reg_lambda": 1.0,
    "reg_alpha": 0.0,
    "gamma": 0.0,
    "early_stopping_rounds": 30,
    "n_jobs": -1,
}

# ============================================================
# 1.6 Input contract object
# ============================================================
# native categorical 활성화 여부는 encoding manifest에서 복원된 categorical feature 존재 여부로 train 모듈이 결정한다.
# InputPaths에 encoding manifest를 포함해 train/val/test 단계가 같은 encoding 계약을 사용하게 한다.
INPUT_PATHS = ml_io.InputPaths(
    train_path=TRAIN_PATH,
    val_path=VAL_PATH,
    test_path=TEST_PATH,
    feature_columns_path=FEATURE_COLUMNS_PATH,
    encoding_manifest_path=ENCODING_MANIFEST_PATH,
)

# ============================================================
# 1.7 Configuration preview
# ============================================================
print("=" * 80)
print("EXPORT_EXPERIMENT_ID  :", EXPORT_EXPERIMENT_ID)
print("RUN_ID                :", RUN_ID)
print("MODEL_RUN_ID          :", MODEL_RUN_ID)
print("ARTIFACT_PREFIX       :", ARTIFACT_PREFIX)
print("ML_ARTIFACT_PREFIX    :", ML_ARTIFACT_PREFIX)
print("TRAIN_PATH            :", TRAIN_PATH)
print("VAL_PATH              :", VAL_PATH)
print("FEATURE_COLUMNS_PATH  :", FEATURE_COLUMNS_PATH)
print("ENCODING_MANIFEST_PATH:", ENCODING_MANIFEST_PATH)
print("OUT_DIR               :", OUT_DIR)
print("OPTUNA_OUTPUT_DIR     :", OPTUNA_OUTPUT_DIR)
print("EXPORT_FEATURE_ASSOC     :", EXPORT_FEATURE_ASSOC)
print("EXPORT_PREDICTION_SCORES :", EXPORT_PREDICTION_SCORES)
print("XGB_ACCELERATOR       :", XGB_ACCELERATOR)
print("=" * 80)


EXPORT_EXPERIMENT_ID  : ml_03
RUN_ID                : r00
MODEL_RUN_ID          : d00-fixparam
ARTIFACT_PREFIX       : ml_03__r00
ML_ARTIFACT_PREFIX    : ml_03__r00__d00-fixparam
TRAIN_PATH            : /mnt/d/AML_git/Graph_AML/ml/ml-03/ml_inputs/r00/ml_03__r00_Xy_train.parquet
VAL_PATH              : /mnt/d/AML_git/Graph_AML/ml/ml-03/ml_inputs/r00/ml_03__r00_Xy_val.parquet
FEATURE_COLUMNS_PATH  : /mnt/d/AML_git/Graph_AML/ml/ml-03/ml_inputs/r00/ml_03__r00_fb_output_feature_contract_approve.csv
ENCODING_MANIFEST_PATH: /mnt/d/AML_git/Graph_AML/ml/ml-03/ml_inputs/r00/ml_03__r00_encoding_manifest.json
OUT_DIR               : /mnt/d/AML_git/Graph_AML/ml/ml-03/ml_outputs/r00
OPTUNA_OUTPUT_DIR     : /mnt/d/AML_git/Graph_AML/ml/ml-03/ml_outputs/r00/optuna__d00-fixparam
EXPORT_FEATURE_ASSOC     : True
EXPORT_PREDICTION_SCORES : True
XGB_ACCELERATOR       : auto


# 02_train / validation 실행

In [3]:
# 입력 parquet, 승인 feature CSV, encoding manifest가 모두 존재하는지 먼저 확인한다.
ml_io.require_input_files(INPUT_PATHS, require_test=RUN_FINAL_TEST)
ml_io.print_input_paths(INPUT_PATHS)

if RUN_TUNING:
    optuna_config = ml_optuna.OptunaPipelineConfig(
        train_path=TRAIN_PATH,
        val_path=VAL_PATH,
        feature_columns_path=FEATURE_COLUMNS_PATH,
        tuning_output_dir=OPTUNA_OUTPUT_DIR,
        final_output_dir=OUT_DIR,
        encoding_manifest_path=ENCODING_MANIFEST_PATH,
        model_file_name=MODEL_FILE_NAME,
        feature_columns_file_name=FEATURE_COLUMNS_FILE_NAME,
        train_summary_file_name=TRAIN_SUMMARY_FILE_NAME,
        scores_train_summary_file_name=SCORES_TRAIN_SUMMARY_FILE_NAME,
        feature_importance_file_name=FEATURE_IMPORTANCE_FILE_NAME,
        threshold_file_name=THRESHOLD_FILE_NAME,
        metrics_val_file_name=METRICS_VAL_FILE_NAME,
        confusion_matrix_val_file_name=CONFUSION_MATRIX_VAL_FILE_NAME,
        smoke_trials=OPTUNA_SMOKE_TRIALS,
        full_trials=OPTUNA_FULL_TRIALS,
        selection_metric=OPTUNA_SELECTION_METRIC,
        study_name_prefix=EXPORT_EXPERIMENT_ID,
        seed=SEED,
        n_jobs=XGB_PARAMS["n_jobs"],
        accelerator=XGB_ACCELERATOR,
        overwrite=OVERWRITE_OUTPUTS,
    )
    optuna_result = ml_optuna.run_ml_optuna_pipeline(optuna_config)
    print("best_params_path  :", optuna_result.best_params_path)
    print("pipeline_summary  :", optuna_result.pipeline_summary_path)
    print("threshold_path    :", optuna_result.val_result.threshold_path)
    print("metrics_path      :", optuna_result.val_result.metrics_path)
else:
    # train_config는 train split, validation split, feature mask, native categorical manifest를 한 번에 묶는다.
    train_config = ml_train.XGBTrainConfig(
        train_path=TRAIN_PATH,
        val_path=VAL_PATH,
        feature_columns_path=FEATURE_COLUMNS_PATH,
        output_dir=OUT_DIR,
        model_file_name=MODEL_FILE_NAME,
        feature_columns_file_name=FEATURE_COLUMNS_FILE_NAME,
        train_summary_file_name=TRAIN_SUMMARY_FILE_NAME,
        scores_train_summary_file_name=SCORES_TRAIN_SUMMARY_FILE_NAME,
        feature_importance_file_name=FEATURE_IMPORTANCE_FILE_NAME,
        sample_rows=SAMPLE_ROWS,
        overwrite=OVERWRITE_OUTPUTS,
        seed=SEED,
        encoding_manifest_path=ENCODING_MANIFEST_PATH,
        export_feature_assoc=EXPORT_FEATURE_ASSOC,
        accelerator=XGB_ACCELERATOR,
        **XGB_PARAMS,
    )
    train_result = ml_train.train_xgb(train_config)

    # threshold 선택과 validation metric 산출은 validation split에서만 수행한다.
    val_config = ml_val.ValidationConfig(
        val_path=VAL_PATH,
        output_dir=OUT_DIR,
        model_file_name=MODEL_FILE_NAME,
        feature_columns_file_name=FEATURE_COLUMNS_FILE_NAME,
        train_summary_file_name=TRAIN_SUMMARY_FILE_NAME,
        scores_train_summary_file_name=SCORES_TRAIN_SUMMARY_FILE_NAME,
        threshold_file_name=THRESHOLD_FILE_NAME,
        metrics_file_name=METRICS_VAL_FILE_NAME,
        confusion_matrix_file_name=CONFUSION_MATRIX_VAL_FILE_NAME,
        sample_rows=SAMPLE_ROWS,
        overwrite=OVERWRITE_OUTPUTS,
        encoding_manifest_path=ENCODING_MANIFEST_PATH,
        export_feature_assoc=EXPORT_FEATURE_ASSOC,
        export_prediction_scores=EXPORT_PREDICTION_SCORES,
    )
    val_result = ml_val.validate_xgb(val_config)
    print("train_summary_path        :", train_result.train_summary_path)
    print("scores_train_summary_path :", train_result.scores_train_summary_path)
    print("threshold_path   :", val_result.threshold_path)
    print("metrics_path     :", val_result.metrics_path)


train_path          : /mnt/d/AML_git/Graph_AML/ml/ml-03/ml_inputs/r00/ml_03__r00_Xy_train.parquet
val_path            : /mnt/d/AML_git/Graph_AML/ml/ml-03/ml_inputs/r00/ml_03__r00_Xy_val.parquet
test_path           : /mnt/d/AML_git/Graph_AML/ml/ml-03/ml_inputs/r00/ml_03__r00_Xy_test.parquet
feature_columns_path: /mnt/d/AML_git/Graph_AML/ml/ml-03/ml_inputs/r00/ml_03__r00_fb_output_feature_contract_approve.csv
encoding_manifest_path: /mnt/d/AML_git/Graph_AML/ml/ml-03/ml_inputs/r00/ml_03__r00_encoding_manifest.json
train_summary_path        : /mnt/d/AML_git/Graph_AML/ml/ml-03/ml_outputs/r00/ml_03__r00__d00-fixparam_train_summary.json
scores_train_summary_path : /mnt/d/AML_git/Graph_AML/ml/ml-03/ml_outputs/r00/ml_03__r00__d00-fixparam_scores_train_summary.json
threshold_path   : /mnt/d/AML_git/Graph_AML/ml/ml-03/ml_outputs/r00/ml_03__r00__d00-fixparam_threshold.json
metrics_path     : /mnt/d/AML_git/Graph_AML/ml/ml-03/ml_outputs/r00/ml_03__r00__d00-fixparam_metrics_val.json


# 03_final test 잠금

In [4]:
# final test는 기본 잠금 상태다. 확정된 full artifact가 있을 때만 RUN_FINAL_TEST=True로 전환한다.
if RUN_FINAL_TEST:
    # confirm_final_test=True를 명시해 test 평가가 의도된 실행임을 남긴다.
    test_config = ml_test.TestConfig(
        test_path=TEST_PATH,
        output_dir=OUT_DIR,
        model_file_name=MODEL_FILE_NAME,
        feature_columns_file_name=FEATURE_COLUMNS_FILE_NAME,
        train_summary_file_name=TRAIN_SUMMARY_FILE_NAME,
        threshold_file_name=THRESHOLD_FILE_NAME,
        metrics_file_name=METRICS_TEST_FILE_NAME,
        confusion_matrix_file_name=CONFUSION_MATRIX_TEST_FILE_NAME,
        confirm_final_test=True,
        overwrite=OVERWRITE_OUTPUTS,
        encoding_manifest_path=ENCODING_MANIFEST_PATH,
        export_feature_assoc=EXPORT_FEATURE_ASSOC,
    )
    test_result = ml_test.test_xgb(test_config)
    print("metrics_path:", test_result.metrics_path)
else:
    print("RUN_FINAL_TEST=False: final test locked")


RUN_FINAL_TEST=False: final test locked


# 04_artifact 요약

저장된 train/validation/final-test artifact를 다시 읽어 실제 학습에 사용된 feature 목록과 metric을 확인한다. 이 섹션은 평가를 새로 실행하지 않는다.

In [5]:
import json
from pathlib import Path

import pandas as pd


def _read_json(path: Path) -> dict:
    with Path(path).open(encoding="utf-8") as f:
        return json.load(f)


def _artifact_paths(out_dir: Path) -> dict[str, Path]:
    return {
        "model": out_dir / MODEL_FILE_NAME,
        "feature_columns": out_dir / FEATURE_COLUMNS_FILE_NAME,
        "train_summary": out_dir / TRAIN_SUMMARY_FILE_NAME,
        "scores_train_summary": out_dir / SCORES_TRAIN_SUMMARY_FILE_NAME,
        "feature_importance": out_dir / FEATURE_IMPORTANCE_FILE_NAME,
        "threshold": out_dir / THRESHOLD_FILE_NAME,
        "metrics_val": out_dir / METRICS_VAL_FILE_NAME,
        "confusion_matrix_val": out_dir / CONFUSION_MATRIX_VAL_FILE_NAME,
        "metrics_test": out_dir / METRICS_TEST_FILE_NAME,
        "confusion_matrix_test": out_dir / CONFUSION_MATRIX_TEST_FILE_NAME,
    }


def _load_actual_feature_columns(paths: dict[str, Path]) -> list[str]:
    feature_columns_path = paths["feature_columns"]
    if feature_columns_path.exists():
        return list(_read_json(feature_columns_path)["feature_columns"])

    approve_table = pd.read_csv(FEATURE_COLUMNS_PATH, encoding="utf-8-sig", dtype={"used_in_ml": "string"})
    return approve_table.loc[approve_table["used_in_ml"] == "TRUE", "column_name"].astype(str).tolist()


def _build_feature_summary(paths: dict[str, Path]) -> tuple[pd.DataFrame, pd.DataFrame]:
    actual_features = _load_actual_feature_columns(paths)
    actual_df = pd.DataFrame({
        "feature_order": range(1, len(actual_features) + 1),
        "column_name": actual_features,
    })

    approve_table = pd.read_csv(FEATURE_COLUMNS_PATH, encoding="utf-8-sig", dtype={"used_in_ml": "string"})
    contract_cols = [
        "column_name",
        "used_in_ml",
        "feature_group",
        "dtype",
        "xgb_feature_type",
        "encoding",
        "source_column",
        "excluded_reason",
        "leakage_risk",
        "selection_note",
    ]
    contract_cols = [col for col in contract_cols if col in approve_table.columns]
    feature_summary = actual_df.merge(approve_table[contract_cols], on="column_name", how="left")

    if paths["feature_importance"].exists():
        importance = pd.read_csv(paths["feature_importance"])
        importance = importance.rename(columns={"feature": "column_name"})
        importance_cols = [
            "column_name",
            "rank_by_gain",
            "importance_gain",
            "importance_weight",
            "importance_cover",
        ]
        importance_cols = [col for col in importance_cols if col in importance.columns]
        feature_summary = feature_summary.merge(importance[importance_cols], on="column_name", how="left")

    manifest_features = []
    if ENCODING_MANIFEST_PATH.exists():
        manifest_features = list(_read_json(ENCODING_MANIFEST_PATH).get("feature_columns", []))

    approved_features = approve_table.loc[
        approve_table["used_in_ml"] == "TRUE", "column_name"
    ].astype(str).tolist()
    actual_set = set(actual_features)
    alignment_rows = []
    for feature_name in manifest_features:
        if feature_name not in actual_set:
            alignment_rows.append({
                "column_name": feature_name,
                "source": "manifest_only",
                "used_in_approve_contract": feature_name in approved_features,
                "used_in_trained_model": False,
            })
    for feature_name in approved_features:
        if feature_name not in actual_set:
            alignment_rows.append({
                "column_name": feature_name,
                "source": "approve_only",
                "used_in_approve_contract": True,
                "used_in_trained_model": False,
            })
    alignment_df = pd.DataFrame(alignment_rows)
    return feature_summary, alignment_df


def _build_run_summary(paths: dict[str, Path]) -> pd.DataFrame:
    if not paths["train_summary"].exists():
        return pd.DataFrame([{"status": "missing_train_summary", "path": str(paths["train_summary"])}])

    train_summary = _read_json(paths["train_summary"])
    threshold_payload = _read_json(paths["threshold"]) if paths["threshold"].exists() else {}
    xgb_params = train_summary.get("xgboost_params", {})
    xgb_acceleration = train_summary.get("xgboost_acceleration", {})
    run_metadata = train_summary.get("run_metadata", {})
    return pd.DataFrame([
        {
            "status": "ok",
            "export_experiment_id": run_metadata.get("export_experiment_id"),
            "input_run_id": run_metadata.get("input_run_id"),
            "model_run_id": run_metadata.get("model_run_id"),
            "run_kind": run_metadata.get("run_kind"),
            "sampled": train_summary.get("sampled"),
            "sample_rows": train_summary.get("sample_rows"),
            "feature_count": train_summary.get("feature_count"),
            "categorical_feature_count": len(train_summary.get("categorical_feature_columns", [])),
            "train_rows": train_summary.get("train_rows"),
            "val_rows": train_summary.get("val_rows"),
            "train_positive_ratio": train_summary.get("train_positive_ratio"),
            "scale_pos_weight": train_summary.get("scale_pos_weight"),
            "best_iteration": train_summary.get("best_iteration"),
            "best_score": train_summary.get("best_score"),
            "threshold_strategy": threshold_payload.get("threshold_strategy"),
            "threshold": threshold_payload.get("threshold"),
            "n_estimators": xgb_params.get("n_estimators"),
            "max_depth": xgb_params.get("max_depth"),
            "learning_rate": xgb_params.get("learning_rate"),
            "enable_categorical": xgb_params.get("enable_categorical"),
            "requested_accelerator": xgb_acceleration.get("requested_accelerator"),
            "resolved_accelerator": xgb_acceleration.get("resolved_accelerator"),
            "xgboost_tree_method": xgb_acceleration.get("xgboost_tree_method", xgb_params.get("tree_method")),
            "xgboost_predictor": xgb_acceleration.get("xgboost_predictor", xgb_params.get("predictor")),
            "gpu_fallback_reason": xgb_acceleration.get("gpu_fallback_reason"),
            "training_time_sec": train_summary.get("training_time_sec"),
            "memory_mb": train_summary.get("memory_mb"),
        }
    ])


def _metric_row(name: str, path: Path) -> dict:
    if not path.exists():
        return {"split": name, "status": "missing", "path": str(path)}

    payload = _read_json(path)
    metrics = payload.get("metrics", {})
    label_info = payload.get("label_summary", {})
    score_info = payload.get("score_profile", {})
    runtime_sec = payload.get("runtime_sec", {})
    total_runtime = runtime_sec.get("total_validate_xgb", runtime_sec.get("total_test_xgb"))

    return {
        "split": payload.get("split", name),
        "status": "ok",
        "sampled": payload.get("sampled"),
        "feature_count": payload.get("feature_count"),
        "feature_columns_hash": payload.get("feature_columns_hash"),
        "threshold_strategy": payload.get("threshold_strategy"),
        "threshold": metrics.get("threshold", payload.get("threshold")),
        "f1": metrics.get("f1"),
        "recall": metrics.get("recall"),
        "precision": metrics.get("precision"),
        "average_precision": metrics.get("average_precision"),
        "tp": metrics.get("tp"),
        "fp": metrics.get("fp"),
        "fn": metrics.get("fn"),
        "tn": metrics.get("tn"),
        "positive_count": label_info.get("positive_count"),
        "positive_ratio": label_info.get("positive_ratio"),
        "predicted_positive_count": score_info.get("predicted_positive_count"),
        "predicted_positive_rate": score_info.get("predicted_positive_rate"),
        "memory_mb": payload.get("memory_mb"),
        "total_runtime_sec": total_runtime,
        "path": str(path),
    }


artifact_paths = _artifact_paths(OUT_DIR)
artifact_status = pd.DataFrame([
    {"artifact": name, "exists": path.exists(), "path": str(path)}
    for name, path in artifact_paths.items()
])

feature_summary_df, feature_alignment_df = _build_feature_summary(artifact_paths)
run_summary_df = _build_run_summary(artifact_paths)
metric_summary_df = pd.DataFrame([
    _metric_row("val", artifact_paths["metrics_val"]),
    _metric_row("test", artifact_paths["metrics_test"]),
])
if "threshold_strategy" in metric_summary_df.columns:
    metric_summary_df["threshold_strategy"] = metric_summary_df["threshold_strategy"].astype("object")
if artifact_paths["threshold"].exists() and "threshold_strategy" in metric_summary_df.columns:
    threshold_payload = _read_json(artifact_paths["threshold"])
    metric_summary_df.loc[metric_summary_df["split"] == "val", "threshold_strategy"] = threshold_payload.get("threshold_strategy")

print("OUT_DIR:", OUT_DIR)
print("Actual trained feature count:", len(feature_summary_df))
display(artifact_status)

print("Train/threshold run summary")
display(run_summary_df)

print("Actual trained feature list and metadata")
display(feature_summary_df)

if feature_alignment_df.empty:
    print("No manifest/approve feature mismatch against trained feature_columns.json.")
else:
    print("Features present in manifest or approve contract but not in trained feature_columns.json")
    display(feature_alignment_df)

print("Validation / final test metrics from saved artifacts")
display(metric_summary_df)

if not artifact_paths["metrics_test"].exists():
    print("metrics_test.json not found: final test has not been evaluated for this OUT_DIR.")


OUT_DIR: /mnt/d/AML_git/Graph_AML/ml/ml-03/ml_outputs/r00
Actual trained feature count: 147


,artifact,exists,path
0,model,True,/mnt/d/AML_git/Graph_AML/ml/ml-03/ml_outputs/r...
1,feature_columns,True,/mnt/d/AML_git/Graph_AML/ml/ml-03/ml_outputs/r...
2,train_summary,True,/mnt/d/AML_git/Graph_AML/ml/ml-03/ml_outputs/r...
3,scores_train_summary,True,/mnt/d/AML_git/Graph_AML/ml/ml-03/ml_outputs/r...
4,feature_importance,True,/mnt/d/AML_git/Graph_AML/ml/ml-03/ml_outputs/r...
5,threshold,True,/mnt/d/AML_git/Graph_AML/ml/ml-03/ml_outputs/r...
6,metrics_val,True,/mnt/d/AML_git/Graph_AML/ml/ml-03/ml_outputs/r...
7,confusion_matrix_val,True,/mnt/d/AML_git/Graph_AML/ml/ml-03/ml_outputs/r...
8,metrics_test,False,/mnt/d/AML_git/Graph_AML/ml/ml-03/ml_outputs/r...
9,confusion_matrix_test,False,/mnt/d/AML_git/Graph_AML/ml/ml-03/ml_outputs/r...


Train/threshold run summary


,status,export_experiment_id,input_run_id,model_run_id,run_kind,sampled,sample_rows,feature_count,categorical_feature_count,train_rows,...,max_depth,learning_rate,enable_categorical,requested_accelerator,resolved_accelerator,xgboost_tree_method,xgboost_predictor,gpu_fallback_reason,training_time_sec,memory_mb
0,ok,ml_03,r00,d00-fixparam,None,False,None,147,6,3046186,...,4,0.05,True,auto,cuda,gpu_hist,gpu_predictor,None,27.433158,10961.320312


Actual trained feature list and metadata


,feature_order,column_name,used_in_ml,feature_group,dtype,xgb_feature_type,encoding,source_column,excluded_reason,leakage_risk,selection_note,rank_by_gain,importance_gain,importance_weight,importance_cover
0,1,amount__current__value,TRUE,parent_ml02_selected,float64,q,passthrough,amount__current__value,NaN,inherited_from_ml02_parent_contract,carried forward from ML-02 parent selected ML ...,28,5356.258789,134.0,134393.328125
1,2,amount__current__log1p,TRUE,parent_ml02_selected,float64,q,passthrough,amount__current__log1p,NaN,inherited_from_ml02_parent_contract,carried forward from ML-02 parent selected ML ...,29,5313.764648,16.0,76659.062500
2,3,amount_received__current__value,TRUE,parent_ml02_selected,float64,q,passthrough,amount_received__current__value,NaN,inherited_from_ml02_parent_contract,carried forward from ML-02 parent selected ML ...,14,10970.527344,120.0,153239.250000
3,4,amount_received__current__log1p,TRUE,parent_ml02_selected,float64,q,passthrough,amount_received__current__log1p,NaN,inherited_from_ml02_parent_contract,carried forward from ML-02 parent selected ML ...,7,32252.240234,12.0,215723.953125
4,5,amount__paid_recv_ratio,TRUE,parent_ml02_selected,float64,q,passthrough,amount__paid_recv_ratio,NaN,inherited_from_ml02_parent_contract,carried forward from ML-02 parent selected ML ...,11,14100.625000,1.0,505334.781250
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
142,143,bankfan__receiver__in__from_bank__nunique__w1d,TRUE,bankfan,float32,q,passthrough,bankfan__receiver__in__from_bank__nunique__w1d,NaN,past-only; history_timestamp < current_timesta...,user selected Stage 2 fan-in/fan-out feature,52,3645.541992,2.0,238803.687500
143,144,bankfan__sender__out__to_bank__nunique__w3d,TRUE,bankfan,float32,q,passthrough,bankfan__sender__out__to_bank__nunique__w3d,NaN,past-only; history_timestamp < current_timesta...,user selected Stage 2 fan-in/fan-out feature,73,2698.645752,7.0,158784.453125
144,145,bankfan__receiver__in__from_bank__nunique__w3d,TRUE,bankfan,float32,q,passthrough,bankfan__receiver__in__from_bank__nunique__w3d,NaN,past-only; history_timestamp < current_timesta...,user selected Stage 2 fan-in/fan-out feature,55,3597.328857,18.0,96647.000000
145,146,bankfan__sender__out__to_bank__nunique__w7d,TRUE,bankfan,float32,q,passthrough,bankfan__sender__out__to_bank__nunique__w7d,NaN,past-only; history_timestamp < current_timesta...,user selected Stage 2 fan-in/fan-out feature,141,168.076324,7.0,1262.842651


No manifest/approve feature mismatch against trained feature_columns.json.
Validation / final test metrics from saved artifacts


,split,status,sampled,feature_count,feature_columns_hash,threshold_strategy,threshold,f1,recall,precision,...,fp,fn,tn,positive_count,positive_ratio,predicted_positive_count,predicted_positive_rate,memory_mb,total_runtime_sec,path
0,val,ok,False,147.0,1732087d99d43adf41b49569b6b24b27c65a25f7c3c913...,max_f1,0.978331,0.25299,0.253924,0.252062,...,816.0,808.0,1013734.0,1083.0,0.001066,1091.0,0.001074,4828.578125,37.529912,/mnt/d/AML_git/Graph_AML/ml/ml-03/ml_outputs/r...
1,test,missing,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,/mnt/d/AML_git/Graph_AML/ml/ml-03/ml_outputs/r...


metrics_test.json not found: final test has not been evaluated for this OUT_DIR.
